In [1]:
import pandas as pd

In [4]:
FFA_cape_data = pd.read_csv("./data/bfa_cape.csv")
FFA_panamax_data = pd.read_csv("./data/bfa_panamax.csv")
FFA_supramax_data = pd.read_csv("./data/bfa_supramax.csv")
freight_data =  pd.read_csv("./data/Freight data.csv")

In [9]:
FFA_supramax_data

,GroupDesc,ArchiveDate,RouteIdentifier,RouteAverage,FFADescription
0,BFA Supramax,24-Oct-2016,10TC_SCURMON,7206.0,Oct 16
1,BFA Supramax,24-Oct-2016,10TC_S+1MON,7693.0,Nov 16
2,BFA Supramax,24-Oct-2016,10TC_S+2MON,7231.0,Dec 16
3,BFA Supramax,24-Oct-2016,10TC_S+3MON,5431.0,Jan 17
4,BFA Supramax,24-Oct-2016,10TC_S+4MON,5271.0,Feb 17
...,...,...,...,...,...
46515,BFA Supramax,01-Oct-2025,11TC_S+3CAL,13252.0,Cal 28
46516,BFA Supramax,01-Oct-2025,11TC_S+4CAL,13241.0,Cal 29
46517,BFA Supramax,01-Oct-2025,11TC_S+5CAL,13366.0,Cal 30
46518,BFA Supramax,01-Oct-2025,11TC_S+6CAL,13466.0,Cal 31


In [6]:
freight_data

,Date,ATCC,ATCE,ATDC,ATIC,ATOPEX,ATSPA,ATSPA10,ATTC,BCI,...,UDSPA,UDSPA10,VLTCE,VTCC,VTDC,VTIC,VTOPEX,VTSPA,VTSPA10,VTTC
0,01-04-10,NaN,24132.0,NaN,NaN,NaN,38.136,NaN,NaN,4197.0,...,NaN,NaN,18869.0,NaN,NaN,NaN,NaN,77.194,NaN,NaN
1,01-05-10,NaN,21430.0,NaN,NaN,NaN,NaN,NaN,NaN,4431.0,...,NaN,NaN,18300.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01-06-10,NaN,20403.0,NaN,NaN,NaN,NaN,NaN,NaN,4253.0,...,NaN,NaN,19873.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,01-07-10,NaN,18823.0,NaN,NaN,NaN,NaN,NaN,NaN,3824.0,...,NaN,NaN,25931.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,01-08-10,NaN,18219.0,NaN,NaN,NaN,NaN,NaN,NaN,3733.0,...,NaN,NaN,35449.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3935,08-15-25,NaN,34943.0,NaN,NaN,NaN,63.319,50.563,NaN,3295.0,...,30.518,22.44,36376.0,NaN,NaN,NaN,NaN,114.540,84.063,NaN
3936,08-18-25,NaN,34184.0,NaN,NaN,NaN,NaN,NaN,NaN,3212.0,...,NaN,NaN,38081.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3937,08-19-25,NaN,32557.0,NaN,NaN,NaN,NaN,NaN,NaN,3023.0,...,NaN,NaN,39032.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3938,08-20-25,NaN,33237.0,NaN,NaN,NaN,NaN,NaN,NaN,2867.0,...,NaN,NaN,40913.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
ffa_columns = [
    "5TC_CCURMON", "5TC_C+1MON", "5TC_C+2MON",
    "4TC_PCURMON", "4TC_P+1MON", "4TC_P+2MON",
    "10TC_SCURMON", "10TC_S+1MON", "10TC_S+2MON",
]

ffa_data = pd.concat(
    [
        FFA_cape_data[["ArchiveDate", "RouteIdentifier", "RouteAverage"]],
        FFA_panamax_data[["ArchiveDate", "RouteIdentifier", "RouteAverage"]],
        FFA_supramax_data[["ArchiveDate", "RouteIdentifier", "RouteAverage"]],
    ],
    ignore_index=True,
)
ffa_data = ffa_data[ffa_data["RouteIdentifier"].isin(ffa_columns)].copy()
ffa_data["Date"] = pd.to_datetime(ffa_data["ArchiveDate"], format="%d-%b-%Y")
ffa_data = ffa_data.pivot(
    index="Date", columns="RouteIdentifier", values="RouteAverage"
).reset_index()

spot_data = freight_data[["Date", "C5TC", "P4TC", "S10TC"]].copy()
spot_data["Date"] = pd.to_datetime(spot_data["Date"], format="%m-%d-%y")

FFA_with_Spot = spot_data.merge(ffa_data, on="Date", how="left")
FFA_with_Spot = FFA_with_Spot[
    ["Date", "C5TC", "5TC_CCURMON", "5TC_C+1MON", "5TC_C+2MON",
     "P4TC", "4TC_PCURMON", "4TC_P+1MON", "4TC_P+2MON",
     "S10TC", "10TC_SCURMON", "10TC_S+1MON", "10TC_S+2MON"]
].rename(columns={"Date": "date"})

FFA_with_Spot.to_csv("./data/FFA_with_Spot.csv", index=False, date_format="%Y-%m-%d")
FFA_with_Spot.head()